# Distributed Training — From Concepts to Code

**What you'll learn in this notebook:**
- Why distributed training is necessary (model size vs. GPU memory)
- Key terminology: world_size, rank, local_rank, process groups
- Collective operations: all-reduce, all-gather, broadcast
- DDP (DistributedDataParallel) patterns
- FSDP2 (Fully Sharded Data Parallel) patterns
- DeviceMesh for multi-dimensional parallelism
- DTensor placements (Shard, Replicate, Partial)
- Choosing a parallelism strategy: decision tree
- Pipeline Parallelism schedules

**Prerequisites:** Understanding of basic model training, gradient descent, parameter memory.

**Note:** Actual distributed execution requires multiple processes/GPUs. This notebook explains concepts with diagrams and pseudocode patterns, with small runnable simulations on CPU.

In [ ]:
import torch
import torch.nn as nn
print(f"PyTorch version: {torch.__version__}")

## Why Distributed Training?

Models have grown exponentially, but single-GPU memory has not:

```
Model           Parameters     Memory (fp16)    Single A100 (80GB)?
─────────────── ────────────── ──────────────── ────────────────────
ResNet-50       25M            50 MB            ✓ Easily
BERT-Large      340M           680 MB           ✓ Easily
GPT-2           1.5B           3 GB             ✓ Fine
LLaMA-7B        7B             14 GB            ✓ Tight with optimizer
LLaMA-70B       70B            140 GB           ✗ Need multiple GPUs
GPT-4 (est.)    ~1.8T          ~3.6 TB          ✗ Need a cluster
```

**Three reasons to go distributed:**
1. **Model doesn't fit** — parameters exceed single GPU memory
2. **Training is too slow** — need more compute throughput
3. **Data doesn't fit** — activations/batch too large for one GPU

In [ ]:
# Calculate memory requirements for a model
def model_memory_estimate(num_params_billions, precision="fp16"):
    """Estimate memory needed for training a model."""
    bytes_per_param = {"fp32": 4, "fp16": 2, "bf16": 2, "fp8": 1}[precision]
    
    param_memory_gb = num_params_billions * 1e9 * bytes_per_param / 1e9
    
    # Adam optimizer: 2 states (momentum + variance) in fp32
    optimizer_memory_gb = num_params_billions * 1e9 * 4 * 2 / 1e9
    
    # Gradients (same precision as params)
    gradient_memory_gb = num_params_billions * 1e9 * bytes_per_param / 1e9
    
    total = param_memory_gb + optimizer_memory_gb + gradient_memory_gb
    
    return {
        "params": param_memory_gb,
        "optimizer": optimizer_memory_gb,
        "gradients": gradient_memory_gb,
        "total (without activations)": total,
    }

print("Memory breakdown for training (excluding activations):")
print("=" * 60)
for size in [1.5, 7, 13, 70]:
    mem = model_memory_estimate(size)
    print(f"\n{size}B params (bf16 training):")
    for k, v in mem.items():
        print(f"  {k:30s}: {v:6.1f} GB")
    gpus_needed = max(1, int(mem["total (without activations)"] / 80) + 1)
    print(f"  {'A100-80GB GPUs needed':30s}: ~{gpus_needed}")

## Key Concepts: Ranks and Process Groups

```
┌─── Node 0 (Machine 0) ───────────────────────┐   ┌─── Node 1 (Machine 1) ───────────────────────┐
│                                               │   │                                               │
│  ┌─────────┐  ┌─────────┐  ┌─────────┐      │   │  ┌─────────┐  ┌─────────┐  ┌─────────┐      │
│  │ GPU 0   │  │ GPU 1   │  │ GPU 2   │  ... │   │  │ GPU 0   │  │ GPU 1   │  │ GPU 2   │  ... │
│  │ rank=0  │  │ rank=1  │  │ rank=2  │      │   │  │ rank=4  │  │ rank=5  │  │ rank=6  │      │
│  │local=0  │  │local=1  │  │local=2  │      │   │  │local=0  │  │local=1  │  │local=2  │      │
│  └─────────┘  └─────────┘  └─────────┘      │   │  └─────────┘  └─────────┘  └─────────┘      │
│                                               │   │                                               │
└───────────────────────────────────────────────┘   └───────────────────────────────────────────────┘
```

- **world_size**: Total number of processes (GPUs) in the job
- **rank**: Global unique ID of each process (0 to world_size-1)
- **local_rank**: GPU index within a single machine (0 to num_gpus_per_node-1)
- **Process group**: A subset of ranks that communicate together

## Collective Operations — The Building Blocks

All distributed strategies are built from these primitives:

### All-Reduce
Every process contributes a tensor → all processes get the sum (or mean).

```
Before:          After all-reduce (sum):
Rank 0: [1,2]   Rank 0: [6,8]
Rank 1: [2,3]   Rank 1: [6,8]
Rank 2: [3,3]   Rank 2: [6,8]
```

### All-Gather
Each process has a shard → every process gets the full tensor.

```
Before:             After all-gather:
Rank 0: [A]         Rank 0: [A, B, C]
Rank 1: [B]         Rank 1: [A, B, C]
Rank 2: [C]         Rank 2: [A, B, C]
```

### Reduce-Scatter
Each process contributes a full tensor → each gets a reduced shard.

```
Before:             After reduce-scatter (sum):
Rank 0: [1,2,3]    Rank 0: [6]   (sum of all rank's first element)
Rank 1: [2,3,4]    Rank 1: [8]   (sum of all rank's second element)
Rank 2: [3,3,5]    Rank 2: [12]  (sum of all rank's third element)
```

### Broadcast
One process sends its tensor to all others.

```
Before:             After broadcast (from rank 0):
Rank 0: [5,6]      Rank 0: [5,6]
Rank 1: [?,?]      Rank 1: [5,6]
Rank 2: [?,?]      Rank 2: [5,6]
```

In [ ]:
# Simulate collective operations (on CPU, single process)
def simulate_all_reduce(tensors, op="sum"):
    """Simulate all-reduce: every rank gets the reduced result."""
    if op == "sum":
        result = torch.stack(tensors).sum(dim=0)
    elif op == "mean":
        result = torch.stack(tensors).mean(dim=0)
    return [result.clone() for _ in tensors]

def simulate_all_gather(shards):
    """Simulate all-gather: every rank gets the full tensor."""
    gathered = torch.cat(shards, dim=0)
    return [gathered.clone() for _ in shards]

def simulate_reduce_scatter(tensors, op="sum"):
    """Simulate reduce-scatter: each rank gets one shard of the reduction."""
    stacked = torch.stack(tensors)
    reduced = stacked.sum(dim=0)
    chunk_size = reduced.shape[0] // len(tensors)
    return [reduced[i*chunk_size:(i+1)*chunk_size] for i in range(len(tensors))]

# Demo
print("=== All-Reduce (sum) ===")
before = [torch.tensor([1., 2.]), torch.tensor([3., 4.]), torch.tensor([5., 6.])]
after = simulate_all_reduce(before)
for i in range(3):
    print(f"  Rank {i}: {before[i].tolist()} → {after[i].tolist()}")

print("\n=== All-Gather ===")
shards = [torch.tensor([1., 2.]), torch.tensor([3., 4.]), torch.tensor([5., 6.])]
gathered = simulate_all_gather(shards)
for i in range(3):
    print(f"  Rank {i}: {shards[i].tolist()} → {gathered[i].tolist()}")

print("\n=== Reduce-Scatter (sum) ===")
before = [torch.tensor([1., 2., 3.]), torch.tensor([4., 5., 6.]), torch.tensor([7., 8., 9.])]
after = simulate_reduce_scatter(before)
for i in range(3):
    print(f"  Rank {i}: {before[i].tolist()} → {after[i].tolist()}")

In [ ]:
# Simulate how DDP gradient sync works
torch.manual_seed(42)

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 2, bias=False)
    
    def forward(self, x):
        return self.linear(x)

# Simulate 3 "GPUs" each processing different data
world_size = 3
models = [SimpleNet() for _ in range(world_size)]

# Ensure all start with same weights (DDP initialization)
with torch.no_grad():
    for m in models[1:]:
        m.linear.weight.copy_(models[0].linear.weight)

# Each "GPU" processes different data and computes local gradients
data_per_gpu = [torch.randn(2, 4) for _ in range(world_size)]
for i, (model, data) in enumerate(zip(models, data_per_gpu)):
    loss = model(data).sum()
    loss.backward()
    print(f"GPU {i} local gradient:\n  {model.linear.weight.grad}")

# All-reduce: average gradients across GPUs
all_grads = [m.linear.weight.grad for m in models]
avg_grad = simulate_all_reduce(all_grads, op="sum")
avg_grad = [g / world_size for g in avg_grad]

print(f"\nAfter all-reduce (averaged):")
print(f"  All GPUs get: {avg_grad[0]}")
print(f"\nThis is exactly what DDP does automatically during backward()!")

## DDP — DistributedDataParallel

DDP is the simplest distributed strategy. Each GPU has a **full copy** of the model and processes different data. Gradients are synchronized via all-reduce after backward.

```
GPU 0: Full model copy    GPU 1: Full model copy    GPU 2: Full model copy
     ↓ (different data)        ↓ (different data)        ↓ (different data)
     Forward pass              Forward pass              Forward pass
     ↓                         ↓                         ↓
     Backward pass             Backward pass             Backward pass
     ↓                         ↓                         ↓
     ←──── All-Reduce gradients across all GPUs ────→
     ↓                         ↓                         ↓
     Optimizer step            Optimizer step            Optimizer step
     (all GPUs now have identical parameters)
```

**Gradient bucketing**: DDP doesn't wait for all gradients — it groups them into buckets and starts all-reduce as soon as each bucket is ready (overlaps communication with computation).

In [ ]:
# DDP code pattern (conceptual — requires multiple processes to actually run)
ddp_code = """
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

# 1. Initialize process group
dist.init_process_group(backend="nccl")  # NCCL for GPU, gloo for CPU
rank = dist.get_rank()
world_size = dist.get_world_size()

# 2. Create model on the correct GPU
torch.cuda.set_device(rank)
model = MyModel().to(rank)
model = DDP(model, device_ids=[rank])

# 3. Use DistributedSampler for data
sampler = torch.utils.data.distributed.DistributedSampler(
    dataset, num_replicas=world_size, rank=rank, shuffle=True
)
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

# 4. Training loop (looks almost identical to single-GPU!)
for epoch in range(num_epochs):
    sampler.set_epoch(epoch)  # Important: ensures different shuffling each epoch
    for batch in loader:
        optimizer.zero_grad()
        loss = model(batch).sum()
        loss.backward()           # DDP handles gradient sync automatically!
        optimizer.step()

dist.destroy_process_group()
"""
print(ddp_code)
print("Key points:")
print("  • Each process runs this SAME script with different rank")
print("  • DDP auto-syncs gradients during backward()")
print("  • DistributedSampler ensures each GPU sees different data")
print("  • Effective batch size = batch_size × world_size")

## FSDP2 — Fully Sharded Data Parallel

DDP replicates the entire model on every GPU — wasteful when models are large. FSDP **shards** parameters, gradients, and optimizer states across GPUs. Each GPU only stores 1/N of the model.

```
DDP (each GPU has everything):      FSDP (sharded across GPUs):
┌────────────────────────┐           ┌────────────────────────┐
│ GPU 0: Full params     │           │ GPU 0: Params shard 0  │
│        Full optimizer  │           │        Optim shard 0   │
│        Full gradients  │           │        Grad shard 0    │
├────────────────────────┤           ├────────────────────────┤
│ GPU 1: Full params     │           │ GPU 1: Params shard 1  │
│        Full optimizer  │           │        Optim shard 1   │
│        Full gradients  │           │        Grad shard 1    │
└────────────────────────┘           └────────────────────────┘
Memory per GPU: Full model           Memory per GPU: 1/N of model
```

**FSDP lifecycle** for each layer during forward/backward:
1. **All-gather** parameters (reconstruct full layer from shards)
2. **Compute** (forward or backward pass through this layer)
3. **Discard** the gathered parameters (free memory immediately)
4. After backward: **reduce-scatter** gradients (each GPU gets its shard of gradients)

In [ ]:
# FSDP2 code pattern
fsdp2_code = """
from torch.distributed._composable.fsdp import fully_shard

# Initialize process group
dist.init_process_group(backend="nccl")

# Create model
model = LargeTransformer()

# Apply FSDP2 per-layer (composable API — no wrapper class!)
for layer in model.transformer_layers:
    fully_shard(layer)     # Shard this layer's params across GPUs
fully_shard(model)         # Shard any remaining params (embeddings, etc.)

# Training loop — looks exactly like single-GPU!
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
for batch in dataloader:
    optimizer.zero_grad()
    loss = model(batch).sum()
    loss.backward()
    optimizer.step()
"""
print(fsdp2_code)
print("FSDP2 advantages over FSDP1:")
print("  • Composable API (no DDP-like wrapper)")
print("  • Works with torch.compile")
print("  • Per-parameter sharding (more flexible)")
print("  • Better memory management")

In [ ]:
# Simulate FSDP shard/unshard lifecycle
torch.manual_seed(0)

full_weight = torch.randn(8, 4)  # Full parameter: [8, 4]
world_size = 4

# Shard across GPUs
shards = full_weight.chunk(world_size, dim=0)  # Each GPU gets [2, 4]
print("FSDP Lifecycle Simulation (4 GPUs, weight shape [8,4]):")
print("=" * 50)
print(f"\n1. AT REST: Each GPU stores only its shard")
for i, s in enumerate(shards):
    print(f"   GPU {i}: shape {list(s.shape)} ({s.numel()} params)")

# Before forward: all-gather to reconstruct full parameter
print(f"\n2. BEFORE FORWARD: All-gather to reconstruct full weight")
reconstructed = torch.cat(list(shards), dim=0)
print(f"   All GPUs now have full weight: shape {list(reconstructed.shape)}")
print(f"   Match original: {torch.allclose(full_weight, reconstructed)}")

# After forward: free the full parameter
print(f"\n3. AFTER FORWARD: Free the gathered weight (save memory!)")
print(f"   Memory per GPU back to: {shards[0].numel()} params (not {full_weight.numel()})")

# Memory savings
savings = 1 - (1 / world_size)
print(f"\n   Memory savings: {savings:.0%} per GPU (only store 1/{world_size} of params)")

## DeviceMesh — Multi-Dimensional Parallelism

Real large-scale training combines multiple parallelism strategies. `DeviceMesh` organizes GPUs into a multi-dimensional grid where each dimension handles a different type of parallelism.

### 1D Mesh (e.g., FSDP across all GPUs)
```
DeviceMesh("cuda", [0, 1, 2, 3, 4, 5, 6, 7])
               └─── all 8 GPUs in one group ───┘
```

### 2D Mesh (e.g., FSDP + Tensor Parallel)
```
DeviceMesh("cuda", [[0, 1, 2, 3],     dim 0: FSDP (shard model across rows)
                    [4, 5, 6, 7]])     dim 1: TP (shard layers across columns)
                     
    TP group 0: [0,1,2,3]  — these GPUs split each layer
    TP group 1: [4,5,6,7]  — these GPUs split each layer
    FSDP group 0: [0,4]    — these GPUs shard different layers
    FSDP group 1: [1,5]    ...
    FSDP group 2: [2,6]
    FSDP group 3: [3,7]
```

### 3D Mesh (DP + TP + PP)
```
DeviceMesh("cuda", mesh_3d, dim_names=("dp", "tp", "pp"))
    dp:  Data parallelism (replicate or shard model)
    tp:  Tensor parallelism (split layers across GPUs)
    pp:  Pipeline parallelism (split stages across GPUs)
```

In [ ]:
# Visualize DeviceMesh concepts (without actual distributed runtime)
import itertools

def visualize_mesh(shape, dim_names):
    """Show how GPUs are organized in a mesh."""
    total_gpus = 1
    for s in shape:
        total_gpus *= s
    
    print(f"DeviceMesh: shape={shape}, dims={dim_names}, total GPUs={total_gpus}")
    print()
    
    if len(shape) == 2:
        print(f"  {'':4s}", end="")
        for col in range(shape[1]):
            print(f"  {dim_names[1]}={col}", end="")
        print()
        
        gpu_id = 0
        for row in range(shape[0]):
            print(f"  {dim_names[0]}={row}", end="  ")
            for col in range(shape[1]):
                print(f" GPU {gpu_id:2d}", end=" ")
                gpu_id += 1
            print()
    
    print()
    # Show process groups per dimension
    for dim_idx, name in enumerate(dim_names):
        print(f"  Groups along '{name}' (dim {dim_idx}):")
        # Groups along dim_idx: fix all other dims, vary dim_idx
        other_dims = [range(shape[d]) for d in range(len(shape)) if d != dim_idx]
        for combo in itertools.product(*other_dims):
            group = []
            for i in range(shape[dim_idx]):
                coords = list(combo)
                coords.insert(dim_idx, i)
                # Convert coords to linear GPU id
                gpu_id = 0
                stride = 1
                for d in range(len(shape) - 1, -1, -1):
                    gpu_id += coords[d] * stride
                    stride *= shape[d]
                group.append(gpu_id)
            print(f"    {group}")
    print()

# 2D mesh: FSDP + TP
print("=" * 60)
print("2D Mesh: FSDP (replicate rows) × TP (shard columns)")
print("=" * 60)
visualize_mesh((2, 4), ("fsdp", "tp"))

print("=" * 60)
print("2D Mesh: DP × TP (common for LLM training)")
print("=" * 60)
visualize_mesh((4, 2), ("dp", "tp"))

## DTensor Placements — How Tensors Map to Devices

DTensor extends regular tensors with **placement** information describing how the tensor is distributed across a mesh dimension:

| Placement | Meaning | Example (4 GPUs) |
|-----------|---------|-------------------|
| **Replicate** | Full copy on each GPU | Each GPU has the full [1024, 512] tensor |
| **Shard(dim)** | Split along `dim` | Each GPU has [256, 512] (sharded on dim 0) |
| **Partial** | Each GPU has a partial sum (needs reduce) | Result of local matmul before all-reduce |

In [ ]:
# Simulate DTensor placements
def show_placement(tensor, placement, num_gpus=4):
    """Visualize how a tensor is distributed."""
    print(f"Full tensor shape: {list(tensor.shape)}")
    print(f"Placement: {placement}")
    print()
    
    if placement == "Replicate":
        for i in range(num_gpus):
            print(f"  GPU {i}: shape={list(tensor.shape)} (full copy)")
    elif placement.startswith("Shard"):
        dim = int(placement.split("(")[1].rstrip(")"))
        chunks = tensor.chunk(num_gpus, dim=dim)
        for i, chunk in enumerate(chunks):
            print(f"  GPU {i}: shape={list(chunk.shape)}")
    elif placement == "Partial":
        shard_size = list(tensor.shape)
        for i in range(num_gpus):
            print(f"  GPU {i}: shape={shard_size} (partial sum, needs all-reduce)")
    print()

tensor = torch.randn(8, 16)

print("=== Replicate ===")
show_placement(tensor, "Replicate", num_gpus=4)

print("=== Shard(0) — split rows ===")
show_placement(tensor, "Shard(0)", num_gpus=4)

print("=== Shard(1) — split columns ===")
show_placement(tensor, "Shard(1)", num_gpus=4)

print("DTensor code pattern:")
print("""
from torch.distributed.tensor import DTensor, Shard, Replicate

# Distribute a tensor
dt = DTensor.from_local(local_tensor, device_mesh, placements=[Shard(0)])

# Operations on DTensors automatically handle communication
# e.g., matmul with Shard(1) × Shard(0) → Partial → all-reduce → Replicate
""")

## Choosing a Strategy: Decision Tree

```
How many parameters does your model have?
│
├─ < 1B params: Fits on one GPU
│   └─ Want faster training? → DDP (simplest, near-linear scaling)
│
├─ 1B - 15B params: Tight fit on one GPU
│   ├─ Fits with optimizer? → DDP
│   └─ Doesn't fit? → FSDP (shard optimizer + gradients + params)
│
├─ 15B - 100B params: Doesn't fit on one GPU
│   ├─ Have 1 node (8 GPUs)? → FSDP
│   └─ Have multiple nodes? → FSDP + TP (tensor parallel within node)
│
└─ > 100B params: Need everything
    └─ FSDP + TP + PP (pipeline parallel across nodes)
        └─ Consider: TP within node, PP across nodes, FSDP across TP groups
```

**Rules of thumb:**
- Start with DDP if model fits
- Move to FSDP when memory is the bottleneck
- Add TP when single-layer is too large or communication bandwidth allows
- Add PP for very long models to reduce pipeline bubbles across slow interconnects

In [ ]:
def recommend_strategy(num_params_b, num_gpus, gpu_memory_gb=80):
    """Recommend a distributed training strategy."""
    # Memory per GPU needed for full model (bf16 params + fp32 optimizer)
    param_mem = num_params_b * 2  # bf16
    optim_mem = num_params_b * 8  # fp32 momentum + variance
    grad_mem = num_params_b * 2   # bf16
    total_mem_gb = param_mem + optim_mem + grad_mem
    
    mem_per_gpu_ddp = total_mem_gb  # DDP: full model on each GPU
    mem_per_gpu_fsdp = total_mem_gb / num_gpus  # FSDP: sharded
    
    print(f"Model: {num_params_b}B params | {num_gpus} GPUs ({gpu_memory_gb}GB each)")
    print(f"  Total training memory: {total_mem_gb:.0f} GB")
    print(f"  DDP memory per GPU: {mem_per_gpu_ddp:.0f} GB", end="")
    print(f" {'✓' if mem_per_gpu_ddp < gpu_memory_gb * 0.7 else '✗'}")
    print(f"  FSDP memory per GPU: {mem_per_gpu_fsdp:.1f} GB", end="")
    print(f" {'✓' if mem_per_gpu_fsdp < gpu_memory_gb * 0.7 else '✗'}")
    
    if mem_per_gpu_ddp < gpu_memory_gb * 0.7:
        strategy = "DDP"
    elif mem_per_gpu_fsdp < gpu_memory_gb * 0.7:
        strategy = "FSDP"
    elif mem_per_gpu_fsdp < gpu_memory_gb * 0.7 * 2:
        strategy = "FSDP + TP"
    else:
        strategy = "FSDP + TP + PP"
    
    print(f"  → Recommended: {strategy}")
    print()

recommend_strategy(1.5, num_gpus=4)
recommend_strategy(7, num_gpus=8)
recommend_strategy(70, num_gpus=64)
recommend_strategy(405, num_gpus=512)

## Pipeline Parallelism Schedules

Pipeline parallelism splits the model into stages, with each stage on different GPU(s). The challenge: naive execution has a **bubble** where GPUs are idle.

Different schedules minimize the bubble:

### GPipe (Synchronous)
```
Time →→→
GPU 0 (stage 0): [F1][F2][F3][F4][  ][  ][  ][  ][B4][B3][B2][B1]
GPU 1 (stage 1): [  ][F1][F2][F3][F4][  ][  ][B4][B3][B2][B1][  ]
GPU 2 (stage 2): [  ][  ][F1][F2][F3][F4][B4][B3][B2][B1][  ][  ]

F = forward microbatch, B = backward microbatch
Bubble: GPUs idle waiting for earlier stages
```

### 1F1B (Interleaved)
```
Time →→→
GPU 0: [F1][F2][F3][F4][B1][F5][B2][F6][B3][  ][B4][B5][B6]
GPU 1: [  ][F1][F2][F3][B1][F4][B2][F5][B3][F6][B4][B5][B6]
GPU 2: [  ][  ][F1][F2][B1][F3][B2][F4][B3][F5][B4][B5][B6]

Interleaves forward and backward → smaller bubble!
```

### Comparison Table
```
Schedule     Bubble Size        Memory          Complexity
──────────── ────────────────── ──────────────── ──────────
GPipe        (p-1)/m            High (store all) Low
1F1B         (p-1)/m            Lower            Medium
Interleaved  (p-1)/(m×v)        Lowest           High
Looped       ~0 (near zero)     Moderate         High

p = pipeline stages, m = microbatches, v = virtual stages
```

In [ ]:
# Compute pipeline bubble fraction
def pipeline_efficiency(num_stages, num_microbatches, schedule="gpipe"):
    """Calculate pipeline efficiency (1 - bubble fraction)."""
    if schedule == "gpipe":
        # Bubble: (p-1) idle slots out of total (m + p - 1) forward slots
        bubble_fraction = (num_stages - 1) / (num_microbatches + num_stages - 1)
    elif schedule == "1f1b":
        # Similar bubble to GPipe but better memory
        bubble_fraction = (num_stages - 1) / (num_microbatches + num_stages - 1)
    elif schedule == "interleaved":
        # Virtual stages reduce bubble
        v = 2  # virtual stages per device
        bubble_fraction = (num_stages - 1) / (num_microbatches * v + num_stages - 1)
    
    efficiency = 1 - bubble_fraction
    return efficiency, bubble_fraction

print("Pipeline Efficiency (higher = less wasted compute)")
print("=" * 65)
print(f"{'Stages':<8} {'Microbatches':<14} {'GPipe':<12} {'Interleaved':<12}")
print("-" * 65)

for stages in [4, 8]:
    for microbatches in [4, 8, 16, 32]:
        eff_gpipe, _ = pipeline_efficiency(stages, microbatches, "gpipe")
        eff_inter, _ = pipeline_efficiency(stages, microbatches, "interleaved")
        print(f"{stages:<8} {microbatches:<14} {eff_gpipe:<12.1%} {eff_inter:<12.1%}")

print("\nKey insight: more microbatches → smaller bubble → better efficiency")
print("But more microbatches → smaller per-microbatch batch → less compute efficiency")

## Try It Yourself: Memory Planning for a 7B Model

**Exercise:** Calculate the memory requirements for training a 7B parameter LLaMA-style model and choose an appropriate parallelism strategy.

Consider:
- Parameters in bf16: 2 bytes each
- Adam optimizer states: 8 bytes per parameter (momentum fp32 + variance fp32)
- Gradients in bf16: 2 bytes each
- Activation memory (estimate): ~2x parameter memory for batch_size=1
- Target GPU: A100-80GB

Questions:
1. What's the total memory needed?
2. Can you fit it on 1 GPU?
3. How many GPUs do you need with FSDP?
4. What batch size can you use?

In [ ]:
# Exercise solution
num_params = 7e9  # 7 billion
gpu_memory = 80   # GB

# Memory breakdown
param_memory = num_params * 2 / 1e9         # bf16
gradient_memory = num_params * 2 / 1e9      # bf16
optimizer_memory = num_params * 8 / 1e9     # fp32 Adam (2 states × 4 bytes)
activation_memory = param_memory * 2        # Rough estimate for batch_size=1

total_memory = param_memory + gradient_memory + optimizer_memory + activation_memory

print("=" * 50)
print("Memory Planning: 7B Parameter Model (bf16 training)")
print("=" * 50)
print(f"  Parameters (bf16):    {param_memory:.1f} GB")
print(f"  Gradients (bf16):     {gradient_memory:.1f} GB")
print(f"  Optimizer (fp32):     {optimizer_memory:.1f} GB")
print(f"  Activations (est.):   {activation_memory:.1f} GB")
print(f"  {'─' * 35}")
print(f"  TOTAL:                {total_memory:.1f} GB")
print()

# Strategy analysis
print(f"Single A100 (80GB): {'CAN' if total_memory < gpu_memory else 'CANNOT'} fit")
print()

for num_gpus in [1, 2, 4, 8]:
    # FSDP shards params + optimizer + gradients, not activations
    sharded = (param_memory + gradient_memory + optimizer_memory) / num_gpus
    per_gpu = sharded + activation_memory
    headroom = gpu_memory - per_gpu
    fits = "✓" if per_gpu < gpu_memory else "✗"
    print(f"  FSDP with {num_gpus} GPUs: {per_gpu:.1f} GB/GPU "
          f"(headroom: {headroom:.1f} GB) {fits}")

print()
print("Recommendation:")
print("  • Use FSDP with 8 GPUs (A100-80GB)")
print("  • This gives ~40GB headroom per GPU for larger batch sizes")
print("  • With batch_size=4 per GPU, activations grow ~4x → still fits")
print("  • Effective batch size = 4 × 8 = 32 (good for LLM training)")

## Summary: The Parallelism Zoo

```
┌───────────────────────────────────────────────────────────────────────────┐
│                        PARALLELISM STRATEGIES                              │
├────────────────┬──────────────────────────────────────────────────────────┤
│ Data Parallel  │ Same model on each GPU, different data                   │
│ (DDP)          │ All-reduce gradients after backward                      │
│                │ Scales: throughput (more data/sec)                        │
├────────────────┼──────────────────────────────────────────────────────────┤
│ Fully Sharded  │ Shard params + optimizer + gradients across GPUs         │
│ (FSDP)         │ All-gather before compute, reduce-scatter after          │
│                │ Scales: memory (larger models per GPU)                    │
├────────────────┼──────────────────────────────────────────────────────────┤
│ Tensor         │ Split individual layers across GPUs                      │
│ Parallel (TP)  │ Column/row parallel linear layers                        │
│                │ Scales: single-layer size                                │
├────────────────┼──────────────────────────────────────────────────────────┤
│ Pipeline       │ Split model stages across GPUs                           │
│ Parallel (PP)  │ Microbatch scheduling (GPipe, 1F1B)                     │
│                │ Scales: model depth                                      │
├────────────────┼──────────────────────────────────────────────────────────┤
│ Context        │ Split sequence length across GPUs                        │
│ Parallel (CP)  │ Ring-attention / Ulysses for long contexts              │
│                │ Scales: sequence length                                  │
└────────────────┴──────────────────────────────────────────────────────────┘
```

## Key Takeaways

1. **Distributed training** is needed when models exceed single-GPU memory or training is too slow
2. **Collective ops** (all-reduce, all-gather, reduce-scatter) are the building blocks of all strategies
3. **DDP** is the simplest: replicate model, sync gradients — start here
4. **FSDP** shards everything (params + optim + grads) for memory efficiency — use when model doesn't fit
5. **DeviceMesh** organizes GPUs into dimensions, each handling a different parallelism axis
6. **DTensor** tracks how tensors are distributed (Shard, Replicate, Partial) and auto-inserts communication
7. **Pipeline Parallelism** splits model into stages — bubble fraction decreases with more microbatches
8. **Rule of thumb**: DDP for <10B, FSDP for 10-100B, FSDP+TP+PP for >100B params
9. **Training memory** ≈ 12× params in bytes (bf16 params + bf16 grads + fp32 optimizer × 2)
10. **FSDP2** is the modern API: composable, compile-friendly, per-parameter sharding